# yatayat-vision — plate detector training (Module 5)

Fine-tunes YOLO11n to localize license plates within vehicle-scale crops (see `docs/data-cards/nepal-plates-kaggle.md`). This is a much easier task than Module 1's vehicle detector - a local 2-epoch smoke test on the M1 already hit mAP50 0.948 - so this run mainly buys more epochs and a larger image size, not a rescue from a hard problem.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

Same storage split as Module 1: Drive holds only checkpoints/exports/manifest. The dataset itself is downloaded via `kagglehub` onto Colab's local disk each session (it's ~830MB, small enough that this costs a couple of minutes, not worth persisting).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/yatayat-vision'
RUNS_DIR = f'{DRIVE_ROOT}/runs'
EXPORTS_DIR = f'{DRIVE_ROOT}/plate_exports'
DATA_DIR = '/content/nepal_plates'

import os
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(EXPORTS_DIR, exist_ok=True)

In [ ]:
!pip install -q ultralytics kagglehub

## Build the train/val split

Same logic as `cv-service/scripts/nepal_plates_prep.py` - downloads the dataset via `kagglehub` (no Kaggle account/API key needed for this public dataset, confirmed working anonymously) and writes plain image-path list files rather than copying anything, since the dataset's own `images/`+`labels/` layout already matches what Ultralytics expects.

In [ ]:
%%writefile nepal_plates_prep.py
import argparse
import random
from pathlib import Path

import kagglehub

DATASET_REF = "ishworsubedii/vehicle-number-plate-datasetnepal"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", default="data/nepal_plates")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--val-fraction", type=float, default=0.15)
    args = parser.parse_args()

    dataset_path = Path(kagglehub.dataset_download(DATASET_REF))
    images_dir = dataset_path / "vehicle_number_plate_detection" / "images"
    labels_dir = dataset_path / "vehicle_number_plate_detection" / "labels"

    image_paths = sorted(p for p in images_dir.glob("*.jpg") if (labels_dir / f"{p.stem}.txt").exists())
    print(f"found {len(image_paths)} images with matching labels")

    rng = random.Random(args.seed)
    rng.shuffle(image_paths)
    split_idx = int(len(image_paths) * (1 - args.val_fraction))
    train_paths, val_paths = image_paths[:split_idx], image_paths[split_idx:]

    out_dir = Path(args.out).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "train.txt").write_text("\n".join(str(p) for p in train_paths) + "\n")
    (out_dir / "val.txt").write_text("\n".join(str(p) for p in val_paths) + "\n")
    (out_dir / "data.yaml").write_text(f"path: {out_dir}\ntrain: train.txt\nval: val.txt\nnames:\n  0: plate\n")

    print(f"train: {len(train_paths)} images, val: {len(val_paths)} images")
    print(f"wrote {out_dir}/data.yaml")


if __name__ == "__main__":
    main()


In [ ]:
import os

if os.path.exists(f'{DATA_DIR}/data.yaml'):
    print('split already built this session, reusing it')
else:
    !python nepal_plates_prep.py --out {DATA_DIR}

!cat {DATA_DIR}/data.yaml

## Train

Same resume-on-rerun pattern as Module 1: checkpoints land in Drive, so a dropped Colab session just resumes instead of restarting.

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'plate_detector'
last_checkpoint = f'{RUNS_DIR}/{RUN_NAME}/weights/last.pt'

if os.path.exists(last_checkpoint):
    print('resuming from a previous checkpoint on Drive')
    model = YOLO(last_checkpoint)
    results = model.train(resume=True)
else:
    model = YOLO('yolo11n.pt')
    results = model.train(
        data=f'{DATA_DIR}/data.yaml',
        epochs=50,
        imgsz=640,
        batch=32,
        project=RUNS_DIR,
        name=RUN_NAME,
        exist_ok=True,
    )

## Validate

In [ ]:
best = YOLO(f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt')
metrics = best.val(data=f'{DATA_DIR}/data.yaml')
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

## Export for local (M1) inference

In [ ]:
import shutil

onnx_path = best.export(format='onnx')
coreml_path = best.export(format='coreml')

for path in [onnx_path, coreml_path]:
    dest = f'{EXPORTS_DIR}/{os.path.basename(path)}'
    if os.path.isdir(path):
        shutil.copytree(path, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(path, dest)
    print('saved to', dest)